In [1]:
# ── Imports & Data Loading ────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import faiss
import json
from sklearn.preprocessing import normalize

# Load product aspect vectors
product_vector_df = pd.read_csv("models/absa/product_aspect_vectors.csv")

# Separate product IDs and vectors
product_ids = product_vector_df["product_id"].tolist()
feature_cols = [c for c in product_vector_df.columns if c != "product_id"]
vectors = product_vector_df[feature_cols].values.astype("float32")

# Normalize for cosine similarity
vectors_normalized = normalize(vectors).astype("float32")

print(f"✅ Loaded {len(product_ids)} products")
print(f"✅ Vector dimensions: {vectors.shape[1]}")
print(f"✅ Feature columns: {feature_cols}")

✅ Loaded 691 products
✅ Vector dimensions: 13
✅ Feature columns: ['availability_location', 'beverage', 'brand_quality', 'food_ingredients', 'food_type', 'freshness_aroma', 'nutrition_ingredients', 'packaging_delivery', 'packaging_format', 'price_value', 'quantity_size', 'target_user', 'taste_flavor']


In [2]:
# ── Build FAISS Index ─────────────────────────────────────────────────────────
dimension = vectors_normalized.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity (on normalized vectors)
index.add(vectors_normalized)
print(f"✅ FAISS index built with {index.ntotal} vectors")

# ── Similar Product Search ────────────────────────────────────────────────────
def find_similar_products(product_id, top_k=5):
    if product_id not in product_ids:
        print(f"Product {product_id} not found")
        return []
    
    idx = product_ids.index(product_id)
    query_vec = vectors_normalized[idx].reshape(1, -1)
    
    scores, indices = index.search(query_vec, top_k + 1)  # +1 because it finds itself
    
    results = []
    for score, i in zip(scores[0], indices[0]):
        if i == idx:
            continue  # skip self
        results.append({
            "product_id": product_ids[i],
            "similarity_score": round(float(score), 4)
        })
    
    return results[:top_k]

# ── Test it ───────────────────────────────────────────────────────────────────
test_product = product_ids[0]
print(f"\nFinding similar products to: {test_product}")
similar = find_similar_products(test_product, top_k=5)
for r in similar:
    print(f"  {r['product_id']} — similarity: {r['similarity_score']}")

✅ FAISS index built with 691 vectors

Finding similar products to: B001E4KFG0
  B004AVYUOW — similarity: 0.866
  B00283LQR8 — similarity: 0.75
  B000LRFZE8 — similarity: 0.75
  B000SEJ84M — similarity: 0.75
  B0006GWXYY — similarity: 0.7071


In [3]:
# ── Gap-Fill Recommender ──────────────────────────────────────────────────────
def find_gap_fill_products(product_id, top_k=5, weak_threshold=-0.1):
    if product_id not in product_ids:
        print(f"Product {product_id} not found")
        return []
    
    idx = product_ids.index(product_id)
    query_vec = vectors[idx]  # use unnormalized for aspect value inspection
    
    # Find weak aspects (scored below threshold)
    weak_aspects = [
        feature_cols[i] for i, score in enumerate(query_vec)
        if score < weak_threshold
    ]
    
    if not weak_aspects:
        print(f"No weak aspects found for {product_id} — it scores well on everything!")
        return []
    
    print(f"Weak aspects for {product_id}: {weak_aspects}")
    
    # Score all other products by how well they cover the weak aspects
    gap_scores = []
    for i, pid in enumerate(product_ids):
        if pid == product_id:
            continue
        other_vec = vectors[i]
        # Average score of this product on the weak aspects
        gap_score = np.mean([
            other_vec[feature_cols.index(a)] for a in weak_aspects
        ])
        gap_scores.append((pid, round(float(gap_score), 4)))
    
    # Sort by highest gap score
    gap_scores.sort(key=lambda x: x[1], reverse=True)
    
    return [{"product_id": pid, "gap_score": score} for pid, score in gap_scores[:top_k]]

# ── Test it ───────────────────────────────────────────────────────────────────
test_product = product_ids[0]
print(f"\nGap-fill recommendations for: {test_product}")
gap_results = find_gap_fill_products(test_product, top_k=5)
for r in gap_results:
    print(f"  {r['product_id']} — gap score: {r['gap_score']}")


Gap-fill recommendations for: B001E4KFG0
No weak aspects found for B001E4KFG0 — it scores well on everything!


In [4]:
# Find a product with at least one weak aspect to test with
for pid in product_ids:
    idx = product_ids.index(pid)
    vec = vectors[idx]
    weak = [feature_cols[i] for i, s in enumerate(vec) if s < -0.1]
    if weak:
        print(f"Product: {pid} | Weak aspects: {weak}")
        break

# Test gap fill on that product
print(f"\nGap-fill recommendations for: {pid}")
gap_results = find_gap_fill_products(pid, top_k=5)
for r in gap_results:
    print(f"  {r['product_id']} — gap score: {r['gap_score']}")

Product: B000LQOCH0 | Weak aspects: ['food_type', 'target_user']

Gap-fill recommendations for: B000LQOCH0
Weak aspects for B000LQOCH0: ['food_type', 'target_user']
  B0026Y3YBK — gap score: 1.0
  B003SO503C — gap score: 1.0
  B003YDP5PA — gap score: 1.0
  B002SRAU80 — gap score: 1.0
  B004S0AQHA — gap score: 1.0


In [7]:
valid_product_ids = set(product_ids)
df_filtered = df_reviews[df_reviews["ProductId"].isin(valid_product_ids)].reset_index(drop=True)
print(f"Filtered reviews: {len(df_filtered)}")

Filtered reviews: 4963


In [8]:
# Reset collection to clear anything partially added
chroma_client.delete_collection("product_reviews")
collection = chroma_client.get_or_create_collection(name="product_reviews")

# Add filtered reviews only
BATCH_SIZE = 500
added = 0

for start in range(0, len(df_filtered), BATCH_SIZE):
    batch = df_filtered.iloc[start:start+BATCH_SIZE]
    
    documents = batch["Text"].tolist()
    ids = [str(row["Id"]) for _, row in batch.iterrows()]
    metadatas = [
        {
            "product_id": row["ProductId"],
            "score": float(row["Score"]),
            "summary": str(row["Summary"])
        }
        for _, row in batch.iterrows()
    ]
    
    collection.add(documents=documents, ids=ids, metadatas=metadatas)
    added += len(batch)
    print(f"  Added {added}/{len(df_filtered)} reviews...")

print(f"\n✅ ChromaDB populated with {collection.count()} reviews")

# Test retrieval
test_query = "bad taste and poor quality dog food"
results = collection.query(query_texts=[test_query], n_results=3)

print(f"\nTop 3 reviews for query: '{test_query}'")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Product: {meta['product_id']} | Score: {meta['score']}")
    print(f"     Summary: {meta['summary']}")
    print(f"     Review: {doc[:150]}...")

  Added 500/4963 reviews...
  Added 1000/4963 reviews...
  Added 1500/4963 reviews...
  Added 2000/4963 reviews...
  Added 2500/4963 reviews...
  Added 3000/4963 reviews...
  Added 3500/4963 reviews...
  Added 4000/4963 reviews...
  Added 4500/4963 reviews...
  Added 4963/4963 reviews...

✅ ChromaDB populated with 4963 reviews

Top 3 reviews for query: 'bad taste and poor quality dog food'

[1] Product: B00139TT72 | Score: 5.0
     Summary: My picky dog LOVES this stuff!
     Review: I have a 11 yr old Pomimo (1/2 Pomeranian, 1/2 American Eskimo) who seems to only like human food. Although I'd love to give him human food all the ti...

[2] Product: B001E4KFG0 | Score: 5.0
     Summary: Good Quality Dog Food
     Review: I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than ...

[3] Product: B00139TT72 | Score: 1.0
     Summary: Not what I thought
     Review: As other viewers have pointed ou

In [11]:
# later to do the whole review dataset :

# Use df_reviews instead of df_filtered
# Delete the existing collection first with chroma_client.delete_collection("product_reviews")
# Re-run the batch loop to add all reviews, but now it will only add those that have a matching product_id in the vector dataset. This way we ensure that all reviews in ChromaDB are relevant to the products we have vectors for, enabling better retrieval and recommendation performance.
# import chromadb
# from chromadb.config import Settings

# # ── Setup ChromaDB ────────────────────────────────────────────────────────────
# chroma_client = chromadb.PersistentClient(path="models/stage4/chroma_db")
# collection = chroma_client.get_or_create_collection(name="product_reviews")

# # ── Load original reviews ─────────────────────────────────────────────────────
# df_reviews = pd.read_csv("/Users/kanishkagupta/Downloads/SEM 2_SPRING2026/IS567_TM/project/Sentiment-Intelligence-Platform/Product Reviews.csv")
# df_reviews = df_reviews.reset_index(drop=True)
# df_reviews = df_reviews[["Id", "ProductId", "Score", "Summary", "Text"]].dropna()

# print(f"✅ Loaded {len(df_reviews)} reviews")

# # ── Store reviews in ChromaDB in batches ──────────────────────────────────────
# BATCH_SIZE = 500
# added = 0

# for start in range(0, len(df_reviews), BATCH_SIZE):
#     batch = df_reviews.iloc[start:start+BATCH_SIZE]
    
#     documents = batch["Text"].tolist()
#     ids = [str(row["Id"]) for _, row in batch.iterrows()]
#     metadatas = [
#         {
#             "product_id": row["ProductId"],
#             "score": float(row["Score"]),
#             "summary": str(row["Summary"])
#         }
#         for _, row in batch.iterrows()
#     ]
    
#     collection.add(documents=documents, ids=ids, metadatas=metadatas)
#     added += len(batch)
#     print(f"  Added {added}/{len(df_reviews)} reviews...")

# print(f"\n✅ ChromaDB populated with {collection.count()} reviews")

# # ── Test retrieval ────────────────────────────────────────────────────────────
# test_query = "bad taste and poor quality dog food"
# results = collection.query(query_texts=[test_query], n_results=3)

# print(f"\nTop 3 reviews for query: '{test_query}'")
# for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
#     print(f"\n[{i+1}] Product: {meta['product_id']} | Score: {meta['score']}")
#     print(f"     Summary: {meta['summary']}")
#     print(f"     Review: {doc[:150]}...")

In [5]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
df = pd.read_csv("/Users/kanishkagupta/Downloads/SEM 2_SPRING2026/IS567_TM/project/Sentiment-Intelligence-Platform/Product Reviews.csv")
print(df.columns.tolist())
print(df.head(2))

['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
   Id   ProductId          UserId ProfileName  HelpfulnessNumerator  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW  delmartian                     1   
1   2  B00813GRG4  A1D87F6ZCVE5NK      dll pa                     0   

   HelpfulnessDenominator  Score        Time                Summary  \
0                       1      5  1303862400  Good Quality Dog Food   
1                       0      1  1346976000      Not as Advertised   

                                                Text  
0  I have bought several of the Vitality canned d...  
1  Product arrived labeled as Jumbo Salted Peanut...  


In [7]:
# Get most frequent summary per product as its display name
product_names = (
    df.groupby("ProductId")["Summary"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
product_names.columns = ["ProductId", "display_name"]
product_names["label"] = product_names["ProductId"] + " — " + product_names["display_name"]
product_names.to_csv("models/absa/product_names.csv", index=False)
print(product_names.head(10))

    ProductId                                       display_name  \
0  0006641040                                          A classic   
1  141278509X                                 The best drink mix   
2  2734888454                                      made in china   
3  2841233731                  Great recipe book for my babycook   
4  7310172001  best dog treat-- great for training---  all do...   
5  7310172101  best dog treat-- great for training---  all do...   
6  7800648702                      Delicious Mildly Sweet Cookie   
7  9376674501                                    Terrific Treats   
8  B00002N8SM                          Doesn't catch fruit flies   
9  B00002NCJC                                      thirty bucks?   

                                               label  
0                             0006641040 — A classic  
1                    141278509X — The best drink mix  
2                         2734888454 — made in china  
3     2841233731 — Great recipe